# Chakravyuh v2 Retraining — Safe, Phased, With Early Reward-Hack Detection

**Goal:** retrain the Analyzer LoRA with the anti-collapse v2 reward profile
(FP penalty = −0.8, benign calibration = 0.5, no format reward on benign
flagged as scam) and catch reward hacking **before** burning full training time.

**Runtime:** ~30–45 min on A100. Uses ~8–12 Colab compute units.

**Phased structure with 3 safety gates:**

| Phase | What | If it fails |
|---|---|---|
| 1. Smoke training (≈80 episodes) | Cheap sanity check that training converges + no crashes | Fix config, rerun |
| 2. **Early reward-hack gate** | Eval on 30 bench scenarios — if detection already 100% flat, reward hacking has already started | Stop. Don't waste compute. |
| 3. Full training (≈400 episodes, resumed) | The real run | — |
| 4. **Per-difficulty ramp check** | Eval on all 135 bench + inspect easy/med/hard/novel — uniform 100% = hacked | Keep v1, report v2 failed |
| 5. **Bench eval + threshold sweep** | Publishable metrics for README | — |

**Outputs saved to Drive:** `/content/drive/MyDrive/chakravyuh/analyzer_lora_v2/`
including full trainer_state.json, adapter weights, eval JSONs, and a
diagnostic plot image.

Run cells top to bottom. Stop at any safety gate that fails.


## 1. Setup — Drive, repo, deps, GPU check


In [ ]:
# 1.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 1.2 Clone the latest Chakravyuh repo
import os, subprocess
REPO = "/content/Chakravyuh"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/UjjwalPardeshi/Chakravyuh.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--rebase"], check=True)
os.chdir(REPO)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
# 1.3 Install training dependencies (this is the heavy cell — ~3-5 min)
!pip install -e '.[llm,eval]' -q
!pip install unsloth -q
print("install complete")


In [ ]:
# 1.4 GPU + env check
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total_gb:.1f} GB")
    assert total_gb >= 35, "need ≥40GB VRAM for Qwen2.5-7B bf16 training"
else:
    raise RuntimeError("No GPU detected — switch to A100/L4 runtime")

# Force mock judge (saves Groq rate-limit time; 55s→12s per step)
os.environ.pop("GROQ_API_KEY", None)
os.environ["WANDB_DISABLED"] = "true"   # avoid wandb login prompt; we save to Drive
print("env configured — mock judge, wandb disabled")


## 2. Phase 1 — Smoke Training (≈80 episodes)

The goal here is NOT to produce a good model — it's to verify training runs
end-to-end on this GPU without OOM/tokenizer/adapter errors.  Uses the v2
reward profile from the start so if there's a reward-hacking pattern,
we see it early.

**Takes ~5 min. If this fails, adjust batch/grad_accum before going further.**


In [ ]:
# 2.1 Launch smoke training (80 episodes, save every 20)
import subprocess

SMOKE_OUT = "/content/Chakravyuh/checkpoints/smoke_v2"
cmd = [
    "python", "-m", "training.grpo_analyzer",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--episodes", "80",
    "--output", SMOKE_OUT,
    "--reward-profile", "v2",
    "--lora-rank", "64",
    "--lora-alpha", "128",
    "--batch-size", "2",
    "--grad-accum", "2",
    "--num-generations", "4",
    "--beta", "0.15",
    "--seed", "42",
    "--no-wandb",
]
print("launching:\n ", " ".join(cmd))
print("-" * 70)
rc = subprocess.run(cmd).returncode
print("-" * 70)
assert rc == 0, f"smoke training failed with exit code {rc}"
print("✓ smoke training done. latest checkpoint:")
!ls -la {SMOKE_OUT}


## 3. **SAFETY GATE 1** — Early reward-hack detection

Load the smoke checkpoint and score 30 bench scenarios (15 scams + 15 benign).
If detection is already pinned at 100% *and* FPR is above 25%, the model has
started learning the "always flag" shortcut. **Stop here** — don't waste compute.

A healthy 80-episode smoke should show:
- detection somewhere in 60–90% range
- FPR below 25%
- Some separation between easy vs hard scams


In [ ]:
# 3.1 Inline LoRA scorer — loads base Qwen + attaches adapter, scores N scenarios
import json, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from chakravyuh_env.agents.llm_analyzer import (
    DEFAULT_SYSTEM_PROMPT, DEFAULT_USER_PROMPT_TEMPLATE, parse_analyzer_response,
)

BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
BENCH = Path("/content/Chakravyuh/data/chakravyuh-bench-v0/scenarios.jsonl")

def _find_latest_checkpoint(root: str):
    p = Path(root)
    ckpts = sorted(p.glob("checkpoint-*"), key=lambda x: int(x.name.rsplit("-", 1)[1]))
    return ckpts[-1] if ckpts else p  # fallback to root if final save lives there

def load_scorer(lora_dir: str):
    ckpt = _find_latest_checkpoint(lora_dir)
    print(f"loading adapter from: {ckpt}")
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa",
    )
    model = PeftModel.from_pretrained(model, str(ckpt))
    model.eval()
    return tok, model

def _scenario_text(rec: dict) -> str:
    """Bench scenarios store the scam in `attack_sequence: [{sender, text, ...}]`."""
    parts = [msg["text"] for msg in rec.get("attack_sequence", [])
             if msg.get("sender") == "scammer"]
    return " ".join(parts)

def _scenario_label(rec: dict) -> tuple[bool, str, str]:
    """Return (is_scam, category, difficulty) from ground_truth."""
    gt = rec.get("ground_truth", {})
    return bool(gt.get("is_scam", False)), gt.get("category", "unknown"), gt.get("difficulty", "unknown")

def score_text(tok, model, text: str) -> float:
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user",   "content": DEFAULT_USER_PROMPT_TEMPLATE.format(message=text)},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False, temperature=0.0,
                             pad_token_id=tok.eos_token_id)
    raw = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    # parse_analyzer_response returns (score, signals, explanation)
    score, _signals, _explanation = parse_analyzer_response(raw)
    return float(score)

def sample_bench(n_scam=15, n_benign=15, seed=42):
    import random
    rng = random.Random(seed)
    scams, benigns = [], []
    with open(BENCH) as f:
        for line in f:
            if not line.strip(): continue
            rec = json.loads(line)
            is_scam, _, _ = _scenario_label(rec)
            (scams if is_scam else benigns).append(rec)
    rng.shuffle(scams); rng.shuffle(benigns)
    print(f"bench has {len(scams)} scams + {len(benigns)} benigns total; sampling {n_scam}+{n_benign}")
    return scams[:n_scam] + benigns[:n_benign]

def compute_metrics(results):
    tp = sum(1 for y, s in results if y and s)
    fp = sum(1 for y, s in results if not y and s)
    fn = sum(1 for y, s in results if y and not s)
    tn = sum(1 for y, s in results if not y and not s)
    det = tp / (tp + fn) if (tp + fn) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    f1 = 2 * prec * det / (prec + det) if (prec + det) else 0
    return dict(detection=det, fpr=fpr, precision=prec, f1=f1, n=len(results))

print("scorer helpers loaded")


In [ ]:
# 3.2 Run the early gate
SMOKE_OUT = "/content/Chakravyuh/checkpoints/smoke_v2"
THRESHOLD = 0.55

tok, model = load_scorer(SMOKE_OUT)
sample = sample_bench(n_scam=15, n_benign=15)
print(f"scoring {len(sample)} scenarios with smoke-checkpoint LoRA ...")

results = []  # (is_scam_truth, flagged)
for i, rec in enumerate(sample):
    is_scam, category, difficulty = _scenario_label(rec)
    text = _scenario_text(rec)
    if not text:
        continue
    score = score_text(tok, model, text)
    flagged = score >= THRESHOLD
    results.append((is_scam, flagged))
    marker = "✓" if flagged == is_scam else "✗"
    if i < 10 or i == len(sample) - 1:
        print(f"  [{marker}] score={score:.2f} flagged={flagged}  truth={'scam' if is_scam else 'benign'}"
              f"  cat={category}  diff={difficulty}")

m = compute_metrics(results)
print(f"\nSMOKE EVAL:  detection={m['detection']:.0%}  FPR={m['fpr']:.0%}"
      f"  F1={m['f1']:.2f}  n={m['n']}")

# Gate
hacking = m["detection"] >= 0.99 and m["fpr"] > 0.25
if hacking:
    print("\n🚨 REWARD-HACK FINGERPRINT DETECTED IN SMOKE PHASE.")
    print("   Detection pinned at 100% + FPR >25%.")
    print("   Full training would just deepen the hack. STOP — tune config first.")
    print("   (lower learning rate, reduce benign_ratio, or tighten beta KL term)")
else:
    print("\n✅ smoke eval looks clean — proceed to full training")

# Free the smoke model before resume
del model, tok
torch.cuda.empty_cache()


## 4. Phase 2 — Full Training (≈400 episodes, resume from smoke)

Only run this if Safety Gate 1 passed. Takes ~30 min on A100.
Saves final LoRA + checkpoints to `analyzer_lora_v2/`.


In [ ]:
# 4.1 Full training, resume from the smoke checkpoint
import subprocess

FULL_OUT = "/content/Chakravyuh/checkpoints/analyzer_lora_v2"

# Copy smoke to full output so --resume picks it up
import shutil
if not os.path.exists(FULL_OUT):
    shutil.copytree("/content/Chakravyuh/checkpoints/smoke_v2", FULL_OUT)

cmd = [
    "python", "-m", "training.grpo_analyzer",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--episodes", "700",
    "--output", FULL_OUT,
    "--reward-profile", "v2",
    "--lora-rank", "64",
    "--lora-alpha", "128",
    "--batch-size", "2",
    "--grad-accum", "2",
    "--num-generations", "4",
    "--beta", "0.15",
    "--seed", "42",
    "--no-wandb",
    "--resume",
]
print("launching full training (resume):\n ", " ".join(cmd))
print("-" * 70)
rc = subprocess.run(cmd).returncode
print("-" * 70)
assert rc == 0, f"full training failed rc={rc}"
print("✓ full training done")
!ls {FULL_OUT}


## 5. **SAFETY GATE 2** — Per-difficulty ramp check

After full training, score the full 135-scenario bench and bucket by difficulty.
A healthy LoRA shows a natural ramp: easy > medium > hard ≥ novel (or close).
A reward-hacked LoRA shows **uniform ≈100% across all buckets** — the
fingerprint you saw on v1.

This cell prints the per-bucket numbers and renders the diagnostic bar chart.
If the LoRA bars are all ≈100%, v2 also hacked — keep v1 and report honestly.


In [ ]:
# 5.1 Full bench eval + per-difficulty ramp
FULL_OUT = "/content/Chakravyuh/checkpoints/analyzer_lora_v2"
tok, model = load_scorer(FULL_OUT)

with open(BENCH) as f:
    all_bench = [json.loads(line) for line in f if line.strip()]
print(f"scoring all {len(all_bench)} bench scenarios ...")

per_bucket = {"easy": [], "medium": [], "hard": [], "novel": []}
results_all = []   # (is_scam_truth, flagged, score, category, difficulty)
for i, rec in enumerate(all_bench):
    is_scam, category, difficulty = _scenario_label(rec)
    text = _scenario_text(rec)
    if not text:
        continue
    s = score_text(tok, model, text)
    flagged = s >= 0.55
    results_all.append((is_scam, flagged, s, category, difficulty))
    if is_scam and difficulty in per_bucket:
        per_bucket[difficulty].append((is_scam, flagged))
    if i % 25 == 0 or i == len(all_bench) - 1:
        print(f"  {i+1}/{len(all_bench)}  last score={s:.2f}  diff={difficulty}")

# Overall metrics
m_all = compute_metrics([(y, f) for y, f, *_ in results_all])
print(f"\nFULL BENCH (n={m_all['n']}):")
print(f"  detection={m_all['detection']:.0%}  FPR={m_all['fpr']:.0%}"
      f"  precision={m_all['precision']:.0%}  F1={m_all['f1']:.2f}")

# Per-bucket detection
print("\nPer-difficulty detection (scams only):")
per_difficulty = {}
for b, rows in per_bucket.items():
    if not rows:
        print(f"  {b:8s}: no scams in this bucket"); continue
    det = sum(f for _, f in rows) / len(rows)
    per_difficulty[b] = {"n": len(rows), "detection_rate": det}
    print(f"  {b:8s}: n={len(rows):3d}  detection={det:.0%}")

# Gate
if per_difficulty:
    lo = min(d["detection_rate"] for d in per_difficulty.values())
    hi = max(d["detection_rate"] for d in per_difficulty.values())
    flat = (hi - lo) < 0.10 and hi > 0.95
    if flat:
        print("\n🚨 v2 ALSO LOOKS REWARD-HACKED (flat ≈100% across all difficulty buckets)")
        print("   Recommendation: keep v1, report v2 as an ablation that failed.")
    else:
        print(f"\n✅ v2 shows a healthy detection ramp (spread = {(hi-lo)*100:.0f}pp)")


In [ ]:
# 5.2 Diagnostic bar chart — baseline (scripted) vs LoRA v2
import matplotlib.pyplot as plt
import numpy as np
PLOTS_DIR = Path("/content/Chakravyuh/docs/assets/plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Load scripted baseline per-difficulty
base = json.loads(open("/content/Chakravyuh/logs/mode_c_scripted_n135.json").read())
base_pd = base.get("per_difficulty", {})

BUCKETS = ["easy", "medium", "hard", "novel"]
base_rates = [base_pd.get(b, {}).get("detection_rate", 0) for b in BUCKETS]
v2_rates   = [per_difficulty.get(b, {}).get("detection_rate", 0) for b in BUCKETS]

x = np.arange(len(BUCKETS)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5.2))
ax.bar(x - w/2, base_rates, w, label="Scripted baseline", color="#9aa0a6")
ax.bar(x + w/2, v2_rates,   w, label="LoRA v2 (retrained)", color="#1967d2")
ax.axhline(1.0, color="black", linestyle=":", linewidth=1, alpha=0.5)
ax.set_xticks(x); ax.set_xticklabels([b.capitalize() for b in BUCKETS])
ax.set_ylim(0, 1.15); ax.set_ylabel("Detection rate")
ax.set_title("v2 per-difficulty ramp — healthy shape or still hacked?")
ax.grid(alpha=0.25, linestyle="--"); ax.legend(loc="lower left")
for i, v in enumerate(base_rates): ax.text(i-w/2, v+0.02, f"{v:.0%}", ha="center", fontsize=9)
for i, v in enumerate(v2_rates):   ax.text(i+w/2, v+0.02, f"{v:.0%}", ha="center", fontsize=9, color="#1967d2")

out = PLOTS_DIR / "v2_per_difficulty_check.png"
plt.savefig(out, dpi=160, bbox_inches="tight"); plt.show()
print(f"✓ saved {out}")


## 6. **SAFETY GATE 3** — Threshold sweep

Re-threshold the scores we already computed in Section 5 across
[0.3 … 0.9]. Produces:
- the F1-optimal operating point
- the "best FPR under 15%" operating point
- the full JSON for plotting later

Runs in ~1 second (no re-scoring; pure arithmetic).


In [ ]:
# 6.1 Threshold sweep from the scores we already computed
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

# results_all tuples: (is_scam, flagged, score, category, difficulty)
scored = [(is_scam, score) for is_scam, _flag, score, *_rest in results_all]
sweep = []
for thr in THRESHOLDS:
    tp = sum(1 for y, s in scored if y and s >= thr)
    fp = sum(1 for y, s in scored if not y and s >= thr)
    fn = sum(1 for y, s in scored if y and s < thr)
    tn = sum(1 for y, s in scored if not y and s < thr)
    det = tp / (tp + fn) if (tp + fn) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    f1 = 2 * prec * det / (prec + det) if (prec + det) else 0
    sweep.append({"threshold": thr, "detection_rate": det, "false_positive_rate": fpr,
                  "precision": prec, "f1": f1})

best_f1 = max(sweep, key=lambda r: r["f1"])
fpr_ok = [r for r in sweep if r["false_positive_rate"] <= 0.15]
best_fpr_ok = max(fpr_ok, key=lambda r: r["f1"]) if fpr_ok else None

print("THRESHOLD SWEEP:")
print(f"  Best-F1:           thr={best_f1['threshold']:.2f}  F1={best_f1['f1']:.3f}"
      f"  det={best_f1['detection_rate']:.0%}  FPR={best_f1['false_positive_rate']:.0%}")
if best_fpr_ok:
    print(f"  Best-F1 @FPR≤15%:  thr={best_fpr_ok['threshold']:.2f}  F1={best_fpr_ok['f1']:.3f}"
          f"  det={best_fpr_ok['detection_rate']:.0%}  FPR={best_fpr_ok['false_positive_rate']:.0%}")
else:
    print("  ⚠ no threshold keeps FPR ≤ 15% — model is over-flagging at every operating point")

# Save eval + sweep JSONs
eval_v2 = {
    "lora_v2": {
        **m_all,
        "per_difficulty": per_difficulty,
        "threshold": 0.55,
    },
    "sweep": sweep,
    "best_f1": best_f1,
    "best_f1_under_15fpr": best_fpr_ok,
}
Path("/content/Chakravyuh/logs/eval_v2.json").write_text(json.dumps(eval_v2, indent=2))
print(f"\n✓ wrote logs/eval_v2.json")


## 7. Save everything to Drive

Copies the final LoRA + eval JSONs + diagnostic plot to
`/content/drive/MyDrive/chakravyuh/analyzer_lora_v2/` so the run survives
Colab runtime expiration and you can download locally for the README.


In [ ]:
# 7.1 Save to Drive
import shutil
DRIVE_OUT = "/content/drive/MyDrive/chakravyuh/analyzer_lora_v2"
FULL_OUT = "/content/Chakravyuh/checkpoints/analyzer_lora_v2"

# Copy adapter + checkpoints
if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(FULL_OUT, DRIVE_OUT)

# Copy eval JSONs
LOGS_DRIVE = "/content/drive/MyDrive/chakravyuh"
shutil.copy("/content/Chakravyuh/logs/eval_v2.json", f"{LOGS_DRIVE}/eval_v2.json")
if os.path.exists("/content/Chakravyuh/docs/assets/plots/v2_per_difficulty_check.png"):
    shutil.copy("/content/Chakravyuh/docs/assets/plots/v2_per_difficulty_check.png",
                f"{LOGS_DRIVE}/v2_per_difficulty_check.png")

print(f"✓ saved to {DRIVE_OUT}")
!ls -la {DRIVE_OUT} 2>&1 | head -15
print(f"\n✓ eval_v2.json also at {LOGS_DRIVE}/eval_v2.json")


## 8. Final summary & pitch-ready numbers

Re-prints the key numbers. Paste into your blog post, slides, or README.


In [ ]:
# 8.1 Pitch-ready summary
print("=" * 60)
print("CHAKRAVYUH v2 RETRAIN — FINAL NUMBERS")
print("=" * 60)
print(f"  Overall  detection={m_all['detection']:.0%}  FPR={m_all['fpr']:.0%}"
      f"  precision={m_all['precision']:.0%}  F1={m_all['f1']:.2f}")
print()
print("  Per-difficulty detection:")
for b in BUCKETS:
    if b in per_difficulty:
        print(f"    {b:8s}  {per_difficulty[b]['detection_rate']:.0%}  (n={per_difficulty[b]['n']})")
print()
print(f"  Best operating point:  thr={best_f1['threshold']:.2f}  "
      f"det={best_f1['detection_rate']:.0%}  FPR={best_f1['false_positive_rate']:.0%}  "
      f"F1={best_f1['f1']:.2f}")
print()

v1 = {"detection": 1.0, "fpr": 0.36, "f1": 0.96}  # from eval_v1.json
print("v1 → v2 DELTA:")
print(f"  Detection:  {v1['detection']:.0%} → {m_all['detection']:.0%}   (v1 was 100% but reward-hacked)")
print(f"  FPR:        {v1['fpr']:.0%} → {m_all['fpr']:.0%}   (lower is better)")
print(f"  F1:         {v1['f1']:.2f} → {m_all['f1']:.2f}")
